In [ ]:
#@title Setup
import datetime
import glob
import json
import pandas as pd
import os
from IPython.display import Image
from ysngs import Config, WorkFlow, execCmd, curlDownload
yscfg = Config()
yscfg.setEnvs()
from ysngs.prepare import preparInput

In [ ]:
#@title Select WDL script
prefix = 'mkidx' #@param {"type": "string"}
wdl = f'{prefix}.wdl'
script = os.path.join(os.environ['HYM_SCRIPT'], 'wdl', wdl)
## Make workflow instance
wf = WorkFlow(script)
wf.check()

In [ ]:
#@title Make input(s)
# Set input
#@markdown Select apps to make index
use_hts = True #@param {type: 'boolean'}
#@markdown
use_bwa = False #@param {type: 'boolean'}
#@markdown
use_bowtie = False #@param {type: 'boolean'}
#@markdown
use_gatk = False #@param {type: 'boolean'}
#@markdown
use_star = False #@param {type: 'boolean'}
#@markdown
use_hisat = False #@param {type: 'boolean'}
#@markdown
use_rsem = False #@param {type: 'boolean'}
#@markdown
use_cr = False #@param {type: 'boolean'}

#@markdown Set output directory
out_dir = ''  #@param {type: 'string'}
#@markdown Set name/prefix of product(s)
name = ''  #@param {type: 'string'}
#@markdown Set genomic FASTA if needed
fasta = ''  #@param {type: 'string'}
#@markdown Set GTF/GFF for gene annotation if needed
gtf = ''  #@param {type: 'string'}
#@markdown Set path of required 3rd apps if needed
app_path = ''  #@param {type: 'string'}
#@markdown Set max. thread to use
thread = 16 #@param {type: 'raw'}

## Prepare input 
input_path = preparInput(prefix, {
    'use_hts': use_hts,
    'use_bwa': use_bwa,
    'use_bowtie': use_bowtie,
    'use_gatk': use_gatk,
    'use_star': use_star,
    'use_hisat': use_hisat,
    'use_rsem': use_rsem,
    'use_cr': use_cr,
    
    'out_dir': os.path.join(os.environ['HYM_REF'], out_dir),
    'name': name,
    'fasta': fasta,
    'gtf': gtf,
    'app_path': app_path,
    'thread': thread
})

In [ ]:
#@title Run
inputs = json.load(open(input_path))
for input in inputs:
    now = datetime.datetime.now()
    path = os.path.join(os.environ['HYM_TEMP'], f"{now.strftime('%Y-%m-%d_%H-%M-%S')}.json")
    json.dump(input, open(path, 'w'))
    # Run
    wf.run(path)

In [ ]:
#@title Visualize
## Set image file path
graph_path = os.path.join(os.environ['HYM_LOG'], f"{os.path.split(wf.script)[1].replace('.wdl', '.dot')}")
img_path = graph_path.replace('.dot', '.png')
## Make dot file
wf.graph(graph_path, detailed=True)
## Display
display(Image(img_path))